# Manifest Package Operator Tutorial

This notebook only creates and validates destination-independent manifest packages. It does not publish to ADLS or submit Fabric work.

The operator workflow is split into two tools:

1. **This notebook** discovers and validates videos, calculates hashes, and writes the package without Azure access.
2. **The separate backfill command** previews the package, then publishes and registers work only when explicitly submitted. See the [operator guide](../docs/explicit-backfill.md).

> **Safety:** running every cell only prepares local files. No cell contacts Azure or Fabric.

## 1. Roles and prerequisites

The preparation operator needs source videos and the camera catalog, but no Azure permissions. Transfer the package and unchanged videos to the submission operator when preparation is complete.

Host prerequisites:

- Python 3.10 or newer and `uv`;
- this repository and its installed environment (run `uv sync --extra experiments` before opening the notebook, then select its `.venv` kernel);
- FFmpeg's `ffprobe` on `PATH`.

Do not run `uv sync` or environment-changing `uv run` commands against the notebook's environment while its kernel is running. If the environment has been replaced, restart the kernel, select the current `.venv`, and rerun from step 1.

Never place credentials in the catalog, exception inventory, package, notebook, command history, or logs.

In [1]:
from pathlib import Path
import csv
import hashlib
import json
import subprocess
import sys


def find_repo_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "samples").is_dir():
            return candidate
    raise RuntimeError("Run this notebook from inside the people-counter repository")


REPO_ROOT = find_repo_root()
VIDEO_ROOT = REPO_ROOT
PARTITION_PREFIX = "samples/"
INPUT_DIR = REPO_ROOT / "samples" / "manifest-package-tutorial-inputs"
PACKAGE_DIR = REPO_ROOT / "samples" / "manifest-package-tutorial"
CATALOG_PATH = INPUT_DIR / "camera_catalog.csv"
EXCEPTION_INVENTORY_PATH = INPUT_DIR / "video-inventory-exceptions.csv"

INPUT_DIR.mkdir(parents=True, exist_ok=True)
assert PACKAGE_DIR.parent == REPO_ROOT / "samples"

print(f"Repository: {REPO_ROOT}")
print(f"Video root: {VIDEO_ROOT}")
print(f"Package output: {PACKAGE_DIR}")

Repository: /home/vinny/repos/people-counter
Video root: /home/vinny/repos/people-counter
Package output: /home/vinny/repos/people-counter/samples/manifest-package-tutorial


## 2. Create the camera metadata catalog

Create one effective-dated row for each camera configuration, not one row per video. A row defines:

- the source path prefix used to associate videos with a camera;
- stable camera and location identifiers;
- the IANA camera timezone;
- expected frame dimensions;
- the directed counting line;
- how capture time is resolved; and
- the effective UTC interval for that configuration.

Use `capture_time_source=auto` in most cases. Resolution order is:

1. an optional exception-inventory override;
2. embedded stream/container `creation_time`;
3. the configured filename regular expression.

The counting line direction matters: reversing its endpoints reverses `in` and `out`. Coordinates must fit within the declared frame dimensions.

In [2]:
catalog_fields = [
    "catalog_version", "source_path_prefix", "camera_id", "location_id",
    "camera_timezone", "frame_width", "frame_height",
    "counting_line_x1", "counting_line_y1", "counting_line_x2", "counting_line_y2",
    "capture_time_source", "capture_time_regex", "capture_time_format",
    "effective_from_utc", "effective_to_utc",
]

# The sample clips have different dimensions and effective capture dates.
catalog_rows = [
    {
        "catalog_version": 1, "source_path_prefix": "samples/",
        "camera_id": "sample-camera", "location_id": "sample-location",
        "camera_timezone": "UTC", "frame_width": 3626, "frame_height": 1994,
        "counting_line_x1": 0, "counting_line_y1": 997,
        "counting_line_x2": 3625, "counting_line_y2": 997,
        "capture_time_source": "auto", "capture_time_regex": "",
        "capture_time_format": "", "effective_from_utc": "2023-01-01T00:00:00Z",
        "effective_to_utc": "2023-09-28T00:00:00Z",
    },
    {
        "catalog_version": 1, "source_path_prefix": "samples/",
        "camera_id": "sample-camera", "location_id": "sample-location",
        "camera_timezone": "UTC", "frame_width": 3840, "frame_height": 2160,
        "counting_line_x1": 0, "counting_line_y1": 1080,
        "counting_line_x2": 3839, "counting_line_y2": 1080,
        "capture_time_source": "auto", "capture_time_regex": "",
        "capture_time_format": "", "effective_from_utc": "2023-09-28T00:00:00Z",
        "effective_to_utc": "2023-10-01T00:00:00Z",
    },
    {
        "catalog_version": 1, "source_path_prefix": "samples/",
        "camera_id": "sample-camera", "location_id": "sample-location",
        "camera_timezone": "UTC", "frame_width": 1920, "frame_height": 1080,
        "counting_line_x1": 0, "counting_line_y1": 540,
        "counting_line_x2": 1919, "counting_line_y2": 540,
        "capture_time_source": "auto", "capture_time_regex": "",
        "capture_time_format": "", "effective_from_utc": "2023-10-01T00:00:00Z",
        "effective_to_utc": "",
    },
]

with CATALOG_PATH.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=catalog_fields)
    writer.writeheader()
    writer.writerows(catalog_rows)

print(CATALOG_PATH.read_text(encoding="utf-8"))

catalog_version,source_path_prefix,camera_id,location_id,camera_timezone,frame_width,frame_height,counting_line_x1,counting_line_y1,counting_line_x2,counting_line_y2,capture_time_source,capture_time_regex,capture_time_format,effective_from_utc,effective_to_utc
1,samples/,sample-camera,sample-location,UTC,3626,1994,0,997,3625,997,auto,,,2023-01-01T00:00:00Z,2023-09-28T00:00:00Z
1,samples/,sample-camera,sample-location,UTC,3840,2160,0,1080,3839,1080,auto,,,2023-09-28T00:00:00Z,2023-10-01T00:00:00Z
1,samples/,sample-camera,sample-location,UTC,1920,1080,0,540,1919,540,auto,,,2023-10-01T00:00:00Z,



## 3. Preview source-video metadata

Preparation performs this inspection automatically. The preview below helps an operator understand which timestamps can be derived without manual input.

Do not use filesystem modification time or upload time as recording time. If embedded metadata and filename rules cannot provide a trustworthy timestamp, preparation rejects that video with an actionable reason code.

In [3]:
video_paths = sorted((REPO_ROOT / "samples").glob("*.mp4"))
if not video_paths:
    raise RuntimeError("No sample MP4 videos were found")

for video_path in video_paths:
    completed = subprocess.run(
        [
            "ffprobe", "-v", "error", "-select_streams", "v:0",
            "-show_entries",
            "stream=width,height,duration:stream_tags=creation_time:format=duration:format_tags=creation_time",
            "-of", "json", str(video_path),
        ],
        cwd=REPO_ROOT,
        check=True,
        capture_output=True,
        text=True,
    )
    metadata = json.loads(completed.stdout)
    stream = metadata["streams"][0]
    embedded_time = stream.get("tags", {}).get("creation_time")
    duration = stream.get("duration") or metadata.get("format", {}).get("duration")
    print(
        video_path.relative_to(REPO_ROOT),
        f"{stream['width']}x{stream['height']}",
        f"duration={duration}",
        f"creation_time={embedded_time}",
    )

samples/people_crossing.mp4 3626x1994 duration=68.466667 creation_time=2023-09-26T12:18:09.000000Z
samples/subway.mp4 3840x2160 duration=31.396667 creation_time=2023-09-29T16:03:17.000000Z
samples/three_people_walking.mp4 1920x1080 duration=7.000000 creation_time=None


## 4. Run preparation without exception overrides

The first pass intentionally omits `--inventory`. Two sample videos have embedded timestamps; one does not. Exit code `2` means preparation completed but at least one video was rejected.

An incomplete package is never publishable. Review grouped totals in `summary.json` and individual remediation in `rejection-report.csv`.

In [4]:
prepare_base = [
    sys.executable, "-c",
    "from people_counter.manifest_publisher import prepare_main; raise SystemExit(prepare_main())",
    "--catalog", str(CATALOG_PATH),
    "--video-root", str(VIDEO_ROOT),
    "--partition-prefix", PARTITION_PREFIX,
    "--output-dir", str(PACKAGE_DIR),
    "--max-files", "10",
]

first_pass = subprocess.run(
    prepare_base,
    cwd=REPO_ROOT,
    check=False,
    capture_output=True,
    text=True,
)
print(first_pass.stdout.strip())
if first_pass.stderr.strip():
    print(first_pass.stderr.strip())
if first_pass.returncode not in {0, 2}:
    raise RuntimeError(f"Preparation failed with exit code {first_pass.returncode}")
print(f"Exit code: {first_pass.returncode}")

{"already_published": 0, "catalog_sha256": "eabfd7c1a78a096b95e3b3287f723ab95fac4436a2fe4c92de91186003e61662", "discovered": 3, "generator_version": "0.9.62", "inventory_sha256": null, "planned": 2, "published": 0, "rejected": 1, "rejections_by_camera": {"sample-camera": 1}, "rejections_by_prefix": {"samples/": 1}, "rejections_by_reason": {"CAPTURE_TIME_MISSING": 1}, "total_video_duration_seconds": 99.86333400000001}
Using CPython 3.12.3 interpreter at: /usr/bin/python3.12
Removed virtual environment at: .venv
Creating virtual environment at: .venv
   Building people-counter @ file:///home/vinny/repos/people-counter
      Built people-counter @ file:///home/vinny/repos/people-counter
Installed 48 packages in 83ms
Exit code: 2


In [5]:
summary = json.loads((PACKAGE_DIR / "summary.json").read_text(encoding="utf-8"))
with (PACKAGE_DIR / "rejection-report.csv").open(newline="", encoding="utf-8") as handle:
    rejections = list(csv.DictReader(handle))

print(json.dumps(summary, indent=2, sort_keys=True))
print("\nRejections:")
for rejection in rejections:
    print(
        f"- {rejection['video_relative_path']}: {rejection['reason_code']}\n"
        f"  Explanation: {rejection['explanation']}\n"
        f"  Suggested action: {rejection['suggested_action']}"
    )

{
  "already_published": 0,
  "catalog_sha256": "eabfd7c1a78a096b95e3b3287f723ab95fac4436a2fe4c92de91186003e61662",
  "discovered": 3,
  "generator_version": "0.9.62",
  "inventory_sha256": null,
  "planned": 2,
  "published": 0,
  "rejected": 1,
  "rejections_by_camera": {
    "sample-camera": 1
  },
  "rejections_by_prefix": {
    "samples/": 1
  },
  "rejections_by_reason": {
    "CAPTURE_TIME_MISSING": 1
  },
  "total_video_duration_seconds": 99.86333400000001
}

Rejections:
- samples/three_people_walking.mp4: CAPTURE_TIME_MISSING
  Explanation: Capture time cannot be derived for samples/three_people_walking.mp4
  Suggested action: Add a filename timestamp rule, repair embedded creation_time, or add only this file to the exception inventory.


## 5. Add only exceptional capture-time overrides

Do not manually inventory all videos. The optional inventory exists only for files whose trustworthy capture time cannot be derived by code.

The timestamp below is a tutorial value. In production, obtain it from an authoritative camera or source-system record and include an explicit UTC `Z` or numeric offset.

In [6]:
exception_rows = [
    {
        "video_relative_path": "samples/three_people_walking.mp4",
        "captured_at_utc": "2023-10-01T01:00:00Z",
    }
]

with EXCEPTION_INVENTORY_PATH.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=["video_relative_path", "captured_at_utc"],
    )
    writer.writeheader()
    writer.writerows(exception_rows)

print(EXCEPTION_INVENTORY_PATH.read_text(encoding="utf-8"))

video_relative_path,captured_at_utc
samples/three_people_walking.mp4,2023-10-01T01:00:00Z



In [7]:
complete_pass = subprocess.run(
    [
        *prepare_base,
        "--inventory", str(EXCEPTION_INVENTORY_PATH),
    ],
    cwd=REPO_ROOT,
    check=True,
    capture_output=True,
    text=True,
)
print(complete_pass.stdout.strip())

{"already_published": 0, "catalog_sha256": "eabfd7c1a78a096b95e3b3287f723ab95fac4436a2fe4c92de91186003e61662", "discovered": 3, "generator_version": "0.9.62", "inventory_sha256": "306f89cfe98d4066491c1a48ee56775a3ebb3a24c92915808d0ae652e4752d3b", "planned": 3, "published": 0, "rejected": 0, "rejections_by_camera": {}, "rejections_by_prefix": {}, "rejections_by_reason": {}, "total_video_duration_seconds": 106.86333400000001}


## 6. Inspect the completed manifest package

`manifest-package.json` is the authoritative checksum index. Prepared manifests intentionally omit `video_uri` and `source_etag`; publication adds those storage-dependent values later.

Identity is deterministic:

```text
asset_id      = SHA256(normalized source-relative path)
asset_version = SHA256(video bytes)
```

Changing one byte creates a new asset version. Correcting bytes at the same relative path retains the logical asset ID.

In [8]:
package_index = json.loads(
    (PACKAGE_DIR / "manifest-package.json").read_text(encoding="utf-8")
)
package_summary = json.loads(
    (PACKAGE_DIR / "summary.json").read_text(encoding="utf-8")
)
with (PACKAGE_DIR / "generated-video-inventory.csv").open(
    newline="", encoding="utf-8"
) as handle:
    generated_inventory = list(csv.DictReader(handle))

print("Package files:")
for path in sorted(PACKAGE_DIR.rglob("*")):
    if path.is_file():
        print("-", path.relative_to(PACKAGE_DIR))

print("\nIndex summary:")
print(json.dumps({
    key: package_index[key]
    for key in ("manifest_package_version", "complete", "discovered", "prepared", "rejected")
}, indent=2))
assert package_index["complete"] is True
assert package_index["prepared"] == len(generated_inventory) == 3
assert not list(PACKAGE_DIR.rglob("*.mp4"))

sample_entry = package_index["entries"][0]
sample_prepared_path = PACKAGE_DIR / sample_entry["prepared_manifest_path"]
sample_prepared = json.loads(sample_prepared_path.read_text(encoding="utf-8"))
assert "video_uri" not in sample_prepared
assert "source_etag" not in sample_prepared
print("\nExample prepared manifest:")
print(json.dumps(sample_prepared, indent=2, sort_keys=True))

Package files:
- generated-video-inventory.csv
- manifest-package.json
- preparation.sqlite3
- prepared-manifests/5b300380f4ea792bf57f1ee1ff6a015258b9da63f5ccdd34bdfbcfb8d3564468/d5bd760178a4ae0d429a52c2c46ae6ec567fea2d55608e982293bfc001a8f750.json
- prepared-manifests/c82ea5ff393efd0de50e7d2fa9f0620829b39caee5dd3e5f92b684ffe74e0f9c/4a780417943542a7d0e72d4f91f30751b8fad304f911936ce7c07ee5a49584c5.json
- prepared-manifests/cbc645f15a90197b2fb3c6d43eed72085c0d32f6ed9acd0cb00826ab04e445b9/0e70fed4fcdc59334f16dc1eac1a5f433403af4fb53a348a26d24b07404b58d4.json
- rejection-report.csv
- summary.json

Index summary:
{
  "manifest_package_version": 1,
  "complete": true,
  "discovered": 3,
  "prepared": 3,
  "rejected": 0
}

Example prepared manifest:
{
  "asset_id": "c82ea5ff393efd0de50e7d2fa9f0620829b39caee5dd3e5f92b684ffe74e0f9c",
  "asset_version": "4a780417943542a7d0e72d4f91f30751b8fad304f911936ce7c07ee5a49584c5",
  "camera_id": "sample-camera",
  "camera_timezone": "UTC",
  "capture_time_s

In [9]:
for row in generated_inventory:
    relative_path = row["video_relative_path"]
    video_path = VIDEO_ROOT.joinpath(*relative_path.split("/"))
    expected_asset_id = hashlib.sha256(relative_path.encode("utf-8")).hexdigest()
    video_digest = hashlib.sha256()
    with video_path.open("rb") as source:
        while chunk := source.read(8 * 1024 * 1024):
            video_digest.update(chunk)
    assert row["asset_id"] == expected_asset_id
    assert row["asset_version"] == video_digest.hexdigest()
    assert row["sha256"] == row["asset_version"]
    print(relative_path, row["asset_version"])

print("All generated identities match the source paths and exact video bytes.")

samples/people_crossing.mp4 4a780417943542a7d0e72d4f91f30751b8fad304f911936ce7c07ee5a49584c5
samples/subway.mp4 d5bd760178a4ae0d429a52c2c46ae6ec567fea2d55608e982293bfc001a8f750
samples/three_people_walking.mp4 0e70fed4fcdc59334f16dc1eac1a5f433403af4fb53a348a26d24b07404b58d4
All generated identities match the source paths and exact video bytes.


## 7. Validate the completed package offline

This validation is offline: it reads the package and rehashes every referenced video before any Azure client is created. It catches edited package files, missing videos, changed bytes, unsupported versions, duplicate identities, inconsistent counts, and incomplete preparation.

In [12]:
from people_counter.manifest_publisher import (
    ManifestPackagePublishConfig,
    validate_manifest_package,
)

validation_config = ManifestPackagePublishConfig(
    manifest_package_dir=PACKAGE_DIR,
    video_root=VIDEO_ROOT,
    storage_account="validation-only",
    filesystem="validation-only",
    staging_prefix="staging",
    incoming_prefix="incoming",
    checkpoint_path=PACKAGE_DIR / "unused-publication.sqlite3",
    rejection_report_path=PACKAGE_DIR / "unused-publication-rejections.csv",
    summary_report_path=PACKAGE_DIR / "unused-publication-summary.json",
    chunk_size=8 * 1024 * 1024,
)

validated_package = validate_manifest_package(validation_config)
print(f"Validated {len(validated_package.plans)} videos without Azure access.")

Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "/home/vinny/repos/people-counter/.venv/lib/python3.13/site-packages/IPython/core/interactiveshell.py", line 3823, in run_code
  File "/tmp/ipykernel_1116988/1660913287.py", line 1, in <module>
    from people_counter.manifest_publisher import (
    ...<2 lines>...
    )
  File "/home/vinny/repos/people-counter/src/people_counter/__init__.py", line 4, in <module>
    from people_counter.api import (
    ...<9 lines>...
    )
  File "/home/vinny/repos/people-counter/src/people_counter/api.py", line 7, in <module>
    from people_counter.config import (
    ...<2 lines>...
    )
  File "/home/vinny/repos/people-counter/src/people_counter/config.py", line 9, in <module>
    from people_counter.models import LineCoordinates, RunResult
  File "/home/vinny/repos/people-counter/src/people_counter/models.py", line 5, in <module>
    import numpy as np
ModuleNotFoundError: No module named 'numpy'

During handling of the above exception, another exceptio

## 8. Retain or transfer the package

Publication may happen immediately or later. The publication operator needs:

1. the unchanged source-video directory tree; and
2. the complete manifest package directory.

The absolute path may change, but relative paths under `--video-root` must remain the same. SHA-256 integrity is authoritative: publication rehashes the video files and refuses changed or missing bytes.

Do not edit prepared JSON files or `manifest-package.json`. Correct source/catalog data and rerun preparation instead.

## 9. Package complete: hand off for submission

Stop here. Retain `PACKAGE_DIR` and the unchanged source tree under `VIDEO_ROOT`. The notebook has not uploaded or registered anything.

A deployment administrator generates an approved target/profile catalog. The submission operator runs `python -m people_counter.backfill_operator` and selects `--target` and `--profile` by name, without editing JSON or calculating hashes. Use the [explicit backfill guide](../docs/explicit-backfill.md) to preview the package, then repeat with `--submit` after review.

Do not create registration JSON, calculate partition hashes, or submit SJD jobs by hand. The separate command performs those operations and retains retry evidence outside the package.

## 10. Preparation troubleshooting

| Reason code | Action |
|---|---|
| `CAPTURE_TIME_MISSING` | Repair embedded metadata, add a filename timestamp rule, or add only affected videos to the exception inventory |
| `CAPTURE_TIME_FILENAME_MISMATCH` | Correct the catalog regex/format or supply an authoritative override |
| `NO_CAMERA_MATCH` | Add or correct the catalog `source_path_prefix` |
| `CAMERA_EFFECTIVE_RANGE_MISMATCH` | Correct the timestamp or effective-dated catalog rows |
| `VIDEO_DIMENSION_MISMATCH` | Correct catalog dimensions/counting line or add another effective-dated configuration |
| `VIDEO_PROBE_FAILED` | Validate, repair, or remux the source video |
| `MANIFEST_PACKAGE_INCOMPLETE` | Resolve all preparation rejections and regenerate the package |
| `MANIFEST_PACKAGE_CHECKSUM_MISMATCH` | Recopy or regenerate the package; never edit prepared files |
| `MANIFEST_PACKAGE_VIDEO_MISMATCH` | Restore the exact source bytes or regenerate from the current video tree |

For large backfills, use the grouped rejection counts in `summary.json` to fix the largest issue classes first instead of reviewing console logs one by one.